# Pipeline from a 1D spectrum

Simulate a BSPEC rate map from a BT-Settl template, build a `RampModel`, sync BSPEC calibrations from KOA if needed, then run `Stage0Pipeline` → `Stage1Pipeline` → `Stage2Pipeline` and show the stellar-RV preview.

**Environment variables** (set these in your shell / notebook kernel before running):

| Variable | Required when | Purpose |
|---|---|---|
| `KOA_CALIBRATION_CACHE` | Always | Local KOA calibration cache directory (and SQLite DB) |
| `KECK_OBSERVER_EMAIL` | Syncing / downloading from KOA | Keck Observer account email |
| `KECK_OBSERVER_PASSWORD` | Syncing / downloading from KOA | Keck Observer account password |

If `KOA_CALIBRATION_CACHE` is unset, the setup cell sets it under `OUT`.  
The KOA cell below syncs metadata from remote KOA and downloads any **BSPEC** calibration files that are not already in the cache.


In [ ]:
import os
import hispecdrp

print(f"hispecdrp {hispecdrp.__version__} @ {hispecdrp.__file__}")

# OUTPUT DIRECTORY — update this to your desired output directory
OUT = "/path/to/hispec_pipeline_example"
os.makedirs(OUT, exist_ok=True)

os.environ.setdefault(
    "KOA_CALIBRATION_CACHE",
    os.path.join(OUT, "cals", "KOA_CALIBRATION_CACHE"),
)
print("output:", OUT)
print("KOA_CALIBRATION_CACHE:", os.environ["KOA_CALIBRATION_CACHE"])


## 1. Sync / verify KOA caldb (BSPEC)


In [ ]:
from hispecdrp.calibrations import verify_koa_calibrations

# Requires env vars: KOA_CALIBRATION_CACHE, KECK_OBSERVER_EMAIL, KECK_OBSERVER_PASSWORD
SPECTROGRAPH = "BSPEC"
summary = verify_koa_calibrations(SPECTROGRAPH, ignore_missing_cal_types=["lfc_wavesol"])

print(
    f"{summary['spectrograph']} caldb: {summary['n_records']} record(s); "
    f"downloaded {summary['n_downloaded']}/{summary['n_missing']} missing file(s)"
)
print("types:", summary["cal_types"])
if summary["downloaded"]:
    print("downloaded:", summary["downloaded"])
if summary["failed"]:
    print("failed:", summary["failed"])


## 2. Simulated rate map from a 1D spectrum → `RampModel`

Load a BT-Settl template, densify it, Doppler-shift the densified 1D spectrum by
`radial_velocity_km_s` (`inject_rv_via='spectrum'`, default), paint with ZEMAX, then
build a raw UTR `RampModel`. Stage-0 KOA masters plus sim Stage-1 products
(packaged line-ZP WAVE — **unscaled**, same bake for every RV — traces, fiberflat)
are registered into the local caldb.


In [ ]:
from hispecdrp.simulations.simulate_up_the_ramp import create_ramp_from_spectrum

FIBERS = [1,3]

# Injected stellar RV (km/s). Default path densifies the 1D template, Doppler-shifts
# it, paints with ZEMAX, and attaches the fixed packaged line-ZP WAVE (no WAVE×(1+z)).
INJECTED_RV = 1.0

# Create a UTR map from a controlled seed and template spectrum
# The user can normalize to a specific magnitude (AB mag) and 2MASS filter
ramp, cals, rate_map = create_ramp_from_spectrum(
    OUT,
    spectrograph=SPECTROGRAPH,
    fibers=FIBERS,
    template_filename="lte049.0-2.5-0.0a+0.0.BT-Settl.spec.7.h5",
    mag_ab=3.05,
    filter_band="J",  # BSPEC (yJ)
    seed=0,
    radial_velocity_km_s=INJECTED_RV,
    barycentric_velocity_km_s=0.0,
    inject_rv_via="spectrum",
    sky_filename='mk_skybg_zm_1.0_1.0_ph.dat',
    sky_fiber=3,
)

print(f"fibers: {FIBERS}")
print(f"ramp type: {ramp.meta.model_type}")
print(f"ramp filename: {ramp.meta.filename}")
print(f"ramp data shape: {ramp.data.shape}")
print(f"Stage-0 cals: {sorted(cals)}")
print(f"rate_map shape: {rate_map.shape}")
print(f"wavesol note: {getattr(getattr(cals.get('wavesol'), 'meta', None), 'note', None)}")


## 3. Stage 0 → 1 → 2 via pipeline classes


In [ ]:
from hispecdrp.pipelines import Stage0Pipeline, Stage1Pipeline, Stage2Pipeline

l0 = Stage0Pipeline.call(ramp, output_dir=OUT)

do_fibers = [str(f) for f in FIBERS]
# Fixed packaged line-ZP WAVE from create_ramp_from_spectrum (unscaled for spectrum paint)
stage1_primitives = {
    "spec_extract": {"do_fibers": do_fibers},
    "set_wavesol": {"wavesol": cals["wavesol"]},
}
l1 = Stage1Pipeline.call(l0, output_dir=OUT, primitives=stage1_primitives)

injected_m_s = float(INJECTED_RV) * 1000.0
l2 = Stage2Pipeline.call(
l1,
    output_dir=OUT,
    primitives={
        "blaze_corr": {"do_fibers": do_fibers},
        "stellar_rv": {
            "ccf_input_velocity": injected_m_s,
            "do_fibers": [1],
            "save_preview": True,
            "preview_write_png": False,
        },
        "sky_sub": {"run": True},
    },
)

print(f"L0 type: {l0.meta.model_type}")
print(f"L0 filename: {l0.meta.filename}")
print(f"L1 type: {l1.meta.model_type}")
print(f"L1 filename: {l1.meta.filename}")
print(f"L2 type: {l2.meta.model_type}")
print(f"L2 filename: {l2.meta.filename}")
print(f"injected RV = {injected_m_s:.4f} m/s ({INJECTED_RV:g} km/s)")
for fib in [1]:
    bc = getattr(l2, f"barycorr_fib{fib}", None)
    if bc is not None and len(bc):
        print(f"fiber {fib} BERV = {float(bc['BERV'][0]):.6f} m/s")
    avg = getattr(l2, f"rvccfavg_fib{fib}", None)
    if avg is not None and len(avg):
        rv = float(avg["RV"][0])
        err = float(avg["RV_ERR"][0])
        bias = rv - injected_m_s
        print(f"fiber {fib} ⟨RV⟩ = {rv:.4f} ± {err:.4f} m/s")
        print(f"fiber {fib} bias = {bias:.4f} m/s (recovered − injected)")


## 4. Stellar RV preview


In [ ]:
from IPython.display import HTML, display
from pathlib import Path
import html

preview = l2.get_output_path(output_dir=OUT, suffix="stellar_rv").replace(".fits", "_preview.html")
print(f"{preview} ({os.path.getsize(preview) / 1e6:.2f} MB)")
# iframe srcdoc so Plotly JS runs inside the notebook
display(HTML(
    f'<iframe srcdoc="{html.escape(Path(preview).read_text(encoding="utf-8"), quote=True)}" '
    f'width="100%" height="1400" style="border:0" sandbox="allow-scripts allow-same-origin"></iframe>'
))
